In [1]:
# =============================================================================
# CELL 1 — Imports
# =============================================================================

import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from src.pago_pipeline.ncbi_snapshot import (
    SnapshotMode,
    get_latest_snapshot_manifest_path,
    get_latest_snapshot_protein_uids_path,
    resolve_ncbi_protein_uid_snapshot,
)

In [2]:
# =============================================================================
# CELL 2 — Environment and project root discovery
# =============================================================================

dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your NCBI email and API key at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

project_root_directory = Path(dotenv_path).resolve().parent

print(f"Project root directory: {project_root_directory}")

Project root directory: C:\Programming\Python\pAgo-project


In [3]:
# =============================================================================
# CELL 3 — Snapshot configuration
# =============================================================================

SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create # "SnapshotMode.create_new", "SnapshotMode.reuse_latest", "SnapshotMode.reuse_latest_or_create"
DATASET_NAME = "piwi_all_fields_bacteria"
SEARCH_QUERY = "PIWI[All Fields] AND Bacteria[Organism]"

DEDUPLICATE_IDS = True
SORT_IDS = True

PAGE_SIZE = 1000
MAX_RETRY_ATTEMPTS = 5
REQUEST_DELAY_SECONDS = None  # None -> automatic based on API key presence

snapshot_root_directory = (
        project_root_directory
        / "data"
        / "01-raw"
        / "ncbi"
        / "protein_searches"
        / DATASET_NAME
)

print(f"Snapshot mode:        {SNAPSHOT_MODE}")
print(f"Dataset name:         {DATASET_NAME}")
print(f"Snapshot root:        {snapshot_root_directory}")
print(f"Search query:         {SEARCH_QUERY}")
print(f"Deduplicate IDs:      {DEDUPLICATE_IDS}")
print(f"Sort IDs:             {SORT_IDS}")
print(f"Page size:            {PAGE_SIZE}")
print(f"Max retry attempts:   {MAX_RETRY_ATTEMPTS}")
print(f"Request delay:        {REQUEST_DELAY_SECONDS}")

Snapshot mode:        create_new
Dataset name:         piwi_all_fields_bacteria
Snapshot root:        C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_searches\piwi_all_fields_bacteria
Search query:         PIWI[All Fields] AND Bacteria[Organism]
Deduplicate IDs:      True
Sort IDs:             True
Page size:            1000
Max retry attempts:   5
Request delay:        None


In [4]:
# =============================================================================
# CELL 4 — Resolve active snapshot
# =============================================================================

snapshot_payload = resolve_ncbi_protein_uid_snapshot(
    snapshot_mode=SNAPSHOT_MODE,
    snapshot_root_directory=snapshot_root_directory,
    search_query=SEARCH_QUERY,
    page_size=PAGE_SIZE,
    max_retry_attempts=MAX_RETRY_ATTEMPTS,
    request_delay_seconds=REQUEST_DELAY_SECONDS,
    ncbi_email=os.getenv("NCBI_EMAIL"),
    ncbi_api_key=os.getenv("NCBI_API_KEY"),
    update_latest_directory=True,
)

Found 41324 protein UIDs.
History session: WebEnv=MCID_69cc3ef825ab1ff8020168fa, QueryKey=1.
Extracted 1000 protein UIDs (2.42%).
Extracted 2000 protein UIDs (4.84%).
Extracted 3000 protein UIDs (7.26%).
Extracted 4000 protein UIDs (9.68%).
Extracted 5000 protein UIDs (12.10%).
Extracted 6000 protein UIDs (14.52%).
Extracted 7000 protein UIDs (16.94%).
Extracted 8000 protein UIDs (19.36%).
Extracted 9000 protein UIDs (21.78%).
Extracted 10000 protein UIDs (24.20%).
Extracted 11000 protein UIDs (26.62%).
Extracted 12000 protein UIDs (29.04%).
Extracted 13000 protein UIDs (31.46%).
Extracted 14000 protein UIDs (33.88%).
Extracted 15000 protein UIDs (36.30%).
Extracted 16000 protein UIDs (38.72%).
Extracted 17000 protein UIDs (41.14%).
Extracted 18000 protein UIDs (43.56%).
Extracted 19000 protein UIDs (45.98%).
Extracted 20000 protein UIDs (48.40%).
Extracted 21000 protein UIDs (50.82%).
Extracted 22000 protein UIDs (53.24%).
Extracted 23000 protein UIDs (55.66%).
Extracted 24000 protein

In [5]:
# =============================================================================
# CELL 5 — Extract active snapshot objects for downstream use
# =============================================================================

active_snapshot_directory = snapshot_payload["snapshot_directory"]
active_manifest_file_path = snapshot_payload["manifest_file_path"]
active_protein_uids_file_path = snapshot_payload["protein_uids_file_path"]
active_snapshot_manifest = snapshot_payload["manifest"]
active_protein_uid_list = snapshot_payload["protein_uids"]

In [6]:
# =============================================================================
# CELL 6 — Human-readable summary of the active snapshot
# =============================================================================

print("=== ACTIVE SNAPSHOT ===")
print(f"Snapshot directory: {active_snapshot_directory}")
print(f"Manifest path:      {active_manifest_file_path}")
print(f"Protein IDs path:   {active_protein_uids_file_path}")
print(f"Protein ID count:   {len(active_protein_uid_list)}")
print(f"Retrieved at UTC:   {active_snapshot_manifest.get('retrieved_at_utc')}")
print(f"Search query:       {active_snapshot_manifest.get('search_query')}")
print(f"Translated query:   {active_snapshot_manifest.get('translated_query')}")
print(f"SHA-256:            {active_snapshot_manifest.get('protein_ids_sha256')}")

=== ACTIVE SNAPSHOT ===
Snapshot directory: C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_searches\piwi_all_fields_bacteria\snapshots\2026-03-31T21-39-03Z__q_891f443d754c
Manifest path:      C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_searches\piwi_all_fields_bacteria\snapshots\2026-03-31T21-39-03Z__q_891f443d754c\manifest.json
Protein IDs path:   C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_searches\piwi_all_fields_bacteria\snapshots\2026-03-31T21-39-03Z__q_891f443d754c\protein_uids.txt
Protein ID count:   41324
Retrieved at UTC:   2026-03-31T21:39:03Z
Search query:       PIWI[All Fields] AND Bacteria[Organism]
Translated query:   PIWI[All Fields] AND ("Bacteria"[Organism] OR "Bacteria Latreille et al. 1825"[Organism])
SHA-256:            None


In [7]:
# =============================================================================
# CELL 7 — Convenience paths for the latest copy
# =============================================================================

latest_manifest_file_path = get_latest_snapshot_manifest_path(
    snapshot_root_directory=snapshot_root_directory,
)
latest_protein_ids_file_path = get_latest_snapshot_protein_uids_path(
    snapshot_root_directory=snapshot_root_directory,
)

print("=== LATEST CONVENIENCE COPY ===")
print(f"Latest manifest path:    {latest_manifest_file_path}")
print(f"Latest protein IDs path: {latest_protein_ids_file_path}")

=== LATEST CONVENIENCE COPY ===
Latest manifest path:    C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_searches\piwi_all_fields_bacteria\latest\manifest.json
Latest protein IDs path: C:\Programming\Python\pAgo-project\data\01-raw\ncbi\protein_searches\piwi_all_fields_bacteria\latest\protein_uids.txt


In [8]:
# =============================================================================
# CELL 8 — Aliases intentionally exposed for downstream notebooks
# =============================================================================

protein_ncbi_id_list = active_protein_uid_list
snapshot_manifest = active_snapshot_manifest
snapshot_directory = active_snapshot_directory
manifest_file_path = active_manifest_file_path
protein_ids_file_path = active_protein_uids_file_path

print("Variables exposed for downstream notebooks:")
print("- protein_ncbi_id_list")
print("- snapshot_manifest")
print("- snapshot_directory")
print("- manifest_file_path")
print("- protein_ids_file_path")

Variables exposed for downstream notebooks:
- protein_ncbi_id_list
- snapshot_manifest
- snapshot_directory
- manifest_file_path
- protein_ids_file_path
